# Analyzing Historical Stock & Revenue Data — TSLA vs. GME (Live Fetch)

**Objective:** Explore how **revenue** relates to **stock prices** for Tesla (TSLA) and GameStop (GME) using live data (no CSVs in repo).

**Pipeline**
1) Acquire (prices via API; revenue via `read_html`)  
2) Clean → standardize (`Date`, `Close`, `Revenue`, `Ticker`)  
3) Align revenue (quarterly) to price cadence  
4) Visualize & interpret (price trend, revenue trend, side-by-side)

**Reproducibility**
- All data fetched at runtime.

**Deliverables**
- Notebook with clear sections and figures 


In [2]:
import yfinance as yf
import pandas as pd
import requests
from bs4 import BeautifulSoup
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [3]:
import plotly.io as pio
pio.renderers.default = "iframe"

In [4]:
import warnings
# Ignore all warnings
warnings.filterwarnings("ignore", category=FutureWarning)

## Define Graphing Function


In [5]:
def make_graph(stock_data, revenue_data, stock):
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=("Historical Share Price", "Historical Revenue"), vertical_spacing = .3)
    stock_data_specific = stock_data[stock_data.Date <= '2021-06-14']
    revenue_data_specific = revenue_data[revenue_data.Date <= '2021-04-30']
    fig.add_trace(go.Scatter(x=pd.to_datetime(stock_data_specific.Date, infer_datetime_format=True), y=stock_data_specific.Close.astype("float"), name="Share Price"), row=1, col=1)
    fig.add_trace(go.Scatter(x=pd.to_datetime(revenue_data_specific.Date, infer_datetime_format=True), y=revenue_data_specific.Revenue.astype("float"), name="Revenue"), row=2, col=1)
    fig.update_xaxes(title_text="Date", row=1, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_yaxes(title_text="Price ($US)", row=1, col=1)
    fig.update_yaxes(title_text="Revenue ($US Millions)", row=2, col=1)
    fig.update_layout(showlegend=False,
    height=900,
    title=stock,
    xaxis_rangeslider_visible=True)
    fig.show()
    from IPython.display import display, HTML
    fig_html = fig.to_html()
    display(HTML(fig_html))

## Use yfinance to Extract Stock Data


Using the `Ticker` function to extract `TSLA` stock:


In [43]:
tsla = yf.Ticker("TSLA")

Extract stock information and save it in a dataframe:

In [44]:
tesla_data = tsla.history(period="max")

In [ ]:
tesla_data.reset_index(inplace=True)
tesla_data.head()

## Webscraping to Extract Tesla Revenue Data


In [ ]:
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm"
html_data  = requests.get(url).text
print(html_data)

Using `beautiful_soup` as a parser:


In [33]:
soup = BeautifulSoup(html_data, 'html.parser')

In [35]:
tesla_revenue = pd.DataFrame(columns=["Date", "Revenue"])
for row in tables[table_index].tbody.find_all("tr"):
    col = row.find_all("td")
    date = col[0].text
    Revenue = col[1].text
    tesla_revenue = pd.concat([tesla_revenue,pd.DataFrame({"Date":[date], "Revenue":[Revenue]})], ignore_index=True)    

In [36]:
tesla_revenue["Revenue"] = tesla_revenue['Revenue'].str.replace(',|\$',"",regex=True)

In [37]:
tesla_revenue.dropna(inplace=True)

tesla_revenue = tesla_revenue[tesla_revenue['Revenue'] != ""]

In [ ]:
tesla_revenue.tail()

## Use yfinance to Extract Stock Data


Using the `Ticker` function to extract `GME` stock:


In [48]:
gme = yf.Ticker("GME")

Extract stock information and save it in a dataframe:

In [49]:
gme_data = gme.history(period="max")

In [ ]:
gme_data.reset_index(inplace=True)
gme_data.head()

## Webscraping to Extract GME Revenue Data


In [90]:
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/stock.html"
html_data_2  = requests.get(url).text 

Using `beautiful_soup` as a parser:


In [91]:
soup = BeautifulSoup(html_data_2,"html.parser") 

In [ ]:
tables = soup.find_all('table') 
len(tables)

In [ ]:
for index,table in enumerate(tables):
    if ("Quarterly Revenue" in str(table)):
        table_index = index
print(table_index)

In [ ]:
print(tables[table_index].prettify())

In [95]:
gme_revenue = pd.DataFrame(columns=["Date", "Revenue"])
for row in tables[table_index].tbody.find_all("tr"):
    col = row.find_all("td")
    date = col[0].text
    Revenue = col[1].text
    gme_revenue = pd.concat([gme_revenue,pd.DataFrame({"Date":[date], "Revenue":[Revenue]})], ignore_index=True)    

In [ ]:
gme_revenue["Revenue"] = gme_revenue['Revenue'].str.replace(',|\$',"",regex=True)
gme_revenue

In [ ]:
gme_revenue.tail()

## Tesla Stock Graph


`make_graph` used to graph Tesla Stock Data

In [ ]:
make_graph(tesla_data, tesla_revenue,'Tesla Stock Data')

## GameStop Stock Graph


`make_graph` used to graph GameStop Stock Data.

In [ ]:
make_graph(gme_data, gme_revenue, 'GameStop')



## <h3 align="center"> © IBM Corporation 2020. All rights reserved. <h3/>

<p>
